### Library and Dataset import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy.cluster.hierarchy as sch

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Set plot style
sns.set_style('whitegrid')

In [ ]:
!git clone "https://github.com/GeeksforgeeksDS/21-Days-21-Projects-Dataset"
df = pd.read_csv('/content/21-Days-21-Projects-Dataset/Datasets/Mall_Customers.csv')

print("Dataset loaded successfully.")
print(f"Data shape: {df.shape}")
df.head()

## EDA

In [ ]:
print("Dataset Info:")
df.info()

In [ ]:
df.drop('CustomerID', axis=1, inplace=True)

print("\nDescriptive Statistics:")
print(df.describe())

### Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Distributions of Customer Features', fontsize=16)

sns.histplot(ax=axes[0], data=df, x='Age', kde=True, bins=20, hue='Gender').set_title('Age Distribution')
sns.histplot(ax=axes[1], data=df, x='Annual Income (k$)', kde=True, bins=20, hue='Gender').set_title('Annual Income Distribution')
sns.histplot(ax=axes[2], data=df, x='Spending Score (1-100)', kde=True, bins=20, hue='Gender').set_title('Spending Score Distribution')

plt.show()

### Biariate Analysis

In [ ]:
sns.pairplot(df, vars=['Age', 'Annual Income (k$)', 'Spending Score (1-100)'], hue='Gender', diag_kind='kde')
plt.suptitle('Pair Plot of Customer Features', y=1.02)
plt.show()

### 3D Visualization

In [ ]:
fig = px.scatter_3d(df,
                    x='Annual Income (k$)',
                    y='Spending Score (1-100)',
                    z='Age',
                    color='Gender',
                    title='3D View of Customer Data')
fig.show()

### Segmentation Model 1 - Income & Spending Score

In [ ]:
# 1. Select and scale the features
X1 = df[['Annual Income (k$)', 'Spending Score (1-100)']]
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1)

# 2. Implement the Elbow Method
wcss1 = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X1_scaled)
    wcss1.append(kmeans.inertia_)

# 3. Plot the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss1, marker='o', linestyle='--')
plt.title('Elbow Method for Income-Spending Segmentation')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(range(1, 11))
plt.show()

**Observation:** The "elbow" of the curve is clearly at **k=5**. The WCSS decreases sharply until k=5, and then the rate of decrease flattens out. This confirms our visual intuition from the scatter plot!

In [ ]:
kmeans1 = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['Income_Cluster'] = kmeans1.fit_predict(X1_scaled)

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)',
                hue='Income_Cluster', palette='viridis', s=100, alpha=0.8, edgecolor='black')
plt.title('Customer Segments by Income and Spending')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# Quantitative Persona Analysis
cluster_profiles1 = df.groupby('Income_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
cluster_profiles1['Size'] = df['Income_Cluster'].value_counts()
print("--- Income-Based Cluster Profiles ---")
cluster_profiles1

### Segmentation Model 2 - Age & Spending Score

In [ ]:
# 1. Select and scale the features
X2 = df[['Age', 'Spending Score (1-100)']]
scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)

# 2. Implement the Elbow Method
wcss2 = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X2_scaled)
    wcss2.append(kmeans.inertia_)

# 3. Plot the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss2, marker='o', linestyle='--')
plt.title('Elbow Method for Age-Spending Segmentation')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(range(1, 11))
plt.show()

**Observation:** The elbow is less sharp here, but **k=4** appears to be a reasonable choice.

In [ ]:
# Build and fit the final model for age segmentation
kmeans2 = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
df['Age_Cluster'] = kmeans2.fit_predict(X2_scaled)

# Visualize the new clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)',
                hue='Age_Cluster', palette='magma', s=100, alpha=0.8, edgecolor='black')
plt.title('Customer Segments by Age and Spending')
plt.legend(title='Cluster')
plt.show()

# **ASSIGNMENT**

### Segmentation Model 3 - Gender & Spending Score

In [ ]:
df_gender = df[['Gender', 'Spending Score (1-100)']]
df_gender['Gender'] = df_gender['Gender'].map({'Male': 0, 'Female': 1})

scaler_gender = StandardScaler()
df_gender_scaled = scaler_gender.fit_transform(df_gender)

wcss_gender = []
for k in range(1, 11):
    kmeans_gender = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans_gender.fit(df_gender_scaled)
    wcss_gender.append(kmeans_gender.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss_gender, marker='o', linestyle='--')
plt.title('Elbow Method for Gender-Spending Segmentation')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(range(1, 11))
plt.show()

From the elbow plot, we can observe that the elbow occurs at k=3, suggesting that 3 clusters might be the optimal number for segmenting the data based on Gender and Spending Score.

In [ ]:
kmeans_gender = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
df['Gender_Spending_Cluster'] = kmeans_gender.fit_predict(df_gender_scaled)

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Spending Score (1-100)', y='Gender',
                hue='Gender_Spending_Cluster', palette='viridis', s=100, alpha=0.8, edgecolor='black')
plt.title('Customer Segments by Gender and Spending Score')
plt.legend(title='Cluster')
plt.show()


Cluster 0 (Purple): These are the customers with high spending scores, predominantly represented by females.

Cluster 1 (Green): This cluster contains customers with moderate spending scores, and it includes both males and females.

Cluster 2 (Yellow): This cluster contains customers with low spending scores, primarily consisting of females.

In [ ]:
cluster_profiles_gender_spending = df.groupby('Gender_Spending_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
cluster_profiles_gender_spending['Size'] = df['Gender_Spending_Cluster'].value_counts()
print("--- Gender-Spending Cluster Profiles ---")
print(cluster_profiles_gender_spending)

## Feature Engineering

In [ ]:
df['Spending_Potential'] = df['Annual Income (k$)'] * df['Spending Score (1-100)']
X = df[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Spending_Potential']]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Spending Potential and Other Features Segmentation')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(range(1, 11))
plt.show()

kmeans = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
df['Spending_Potential_Cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Spending_Potential', y='Spending Score (1-100)',
                hue='Spending_Potential_Cluster', palette='magma', s=100, alpha=0.8, edgecolor='black')
plt.title('Customer Segments by Spending Potential and Spending Score')
plt.legend(title='Cluster')
plt.show()

cluster_profiles_spending_potential = df.groupby('Spending_Potential_Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Spending_Potential']].mean().round(2)
cluster_profiles_spending_potential['Size'] = df['Spending_Potential_Cluster'].value_counts()
print("--- Spending Potential-Based Cluster Profiles ---")
print(cluster_profiles_spending_potential)

**Observation:** The elbow is less sharp here, but **k=4** appears to be a reasonable choice.

Cluster 0 - People with 50k - 90k Spnding Potential have higher spending potential 60-100

Cluster 1 - Less potential and less spending score

Cluster 2 - less to moderate spending potential and moderate spending score.

Cluster 3 - Less spending potential but higher spenaind score. (spend thrift)

## Alternative Method - Hierarchical Clustering

In [ ]:
plt.figure(figsize=(20, 10))
dendrogram = sch.dendrogram(sch.linkage(X1_scaled, method='ward'))
plt.title('Dendrogram for Income-Spending Data')
plt.xlabel('Customers')
plt.ylabel('Euclidean Distances')
plt.axhline(y=6, color='r', linestyle='--') # Example cut-off line
plt.show()